# DM_G4_P0007_branch_and_bound

## 0. 학습 범위
- Gate: G4
- Phase: P0007
- Topic: Ch.6 분지한계법 — LP relaxation, upper/lower bound, incumbent, floor/ceil branching, pruning
- Goal: LP relaxation 값을 bound로 해석하고, 반올림이 아니라 branch-and-bound tree로 정수 최적성을 증명한다.
- Source basis: DM_PDF03 우선, DM_PDF04 보조
- 웹 근거 사용: 아니오
- Code mode: hint_only


## 1. 작성 규칙
- 풀이용 노트북에는 최종 선택 조합, 최종 목적값, 완성 풀이를 쓰지 않는다.
- 각 문제는 먼저 변수의 의미와 domain을 분리해서 쓴 뒤 목적함수와 제약식을 작성한다.
- Solver 설명은 변수셀, 목표셀, 제약식 좌변 셀, 우변 셀, int/bin 옵션을 구분해서 적는다.
- 반올림, IF 함수, 자동 0-1 선택 같은 지름길은 정답으로 인정하지 않는다.
- 답안은 각 문제의 빈 답안 템플릿에 작성한다.


## 2. 채점 기준
| 항목 | 배점 | 확인 기준 |
|---|---:|---|
| LP relaxation과 bound 방향 해석 | 25 | max 문제에서 LP relaxation 값은 upper bound, incumbent는 lower bound임을 설명한다. |
| floor/ceil 또는 include/exclude branching | 20 | 소수값 하나를 `<= floor(v)`와 `>= ceil(v)`로 나눈다. |
| pruning 사유 구분 | 25 | infeasible, bound 열세, integer solution 발견을 구분한다. |
| incumbent와 최적성 증명 | 20 | 정수가능해 발견 시 `L` 갱신과 open node 종료 조건을 설명한다. |
| source anchor/node_id/Solver 연결 | 10 | 근거 anchor와 Solver bin/int 설정을 연결한다. |


## 3. Branch-and-Bound 기준표
| 기호 | max 문제 기준 의미 | 절단 판단 |
|---|---|---|
| `Z` | 현 노드 LP relaxation 목적값, 그 노드 하위영역의 upper bound | `Z <= L`이면 절단 |
| `L` | 현재까지 발견한 가장 좋은 정수가능해 값, lower bound | 더 큰 정수해 발견 시 갱신 |
| fractional node | LP 해가 정수조건을 위반한 노드 | bound가 살아 있으면 branch |
| integer node | LP 해가 정수조건을 만족한 노드 | incumbent 갱신 후 절단 |
| infeasible node | LP relaxation도 feasible하지 않은 노드 | 즉시 절단 |


## 4. 문제 세트

### 문제 1. 배송로봇 정수 배치 — LP relaxation과 floor/ceil branching

한 물류센터가 두 종류의 배송로봇 수를 정수로 결정한다. `x1`은 소형 로봇 대수, `x2`는 대형 로봇 대수다.

Maximize

`z = 30x1 + 38x2`

subject to

`9x1 + 4x2 <= 46`

`2x1 + 5x2 <= 28`

`x1, x2 >= 0`, `x1, x2 integer`

LP relaxation을 풀었더니 root node에서 `x1=3.189`, `x2=4.324`, `Z=260.0`이 나왔다고 하자.

요구:
1. LP relaxation 문제를 쓰라.
2. root node의 `Z=260.0`이 max 정수문제에서 어떤 bound인지 설명하라.
3. `x1=3.189`를 기준으로 두 branch를 만들어라.
4. branch `x1<=3`에서 LP relaxation 해가 `x1=3.0`, `x2=4.4`, `Z=257.2`로 나왔다. 다음 branch를 어떻게 만들지 쓰라.
5. branch `x1>=4`에서 LP relaxation 해가 `x1=4.0`, `x2=2.5`, `Z=215.0`으로 나왔다. 이 노드를 바로 버릴 수 있는지, 어떤 정보가 더 필요한지 설명하라.
6. incumbent `L`이 언제 갱신되는지 설명하라.
7. 모든 open node가 없어졌을 때 최적성을 어떻게 증명하는지 쓰라.
8. 오답 진단: “`x1=3.189`, `x2=4.324`를 각각 반올림하면 된다.” 왜 틀렸는지 설명하라.

node_id: `n_DM_PDF03.branch_and_bound`, `n_DM_PDF03.lp_relaxation`, `n_DM_PDF03.upper_lower_bound`, `n_DM_PDF03.branching_floor_ceil`, `n_DM_PDF03.pruning_rules`, `n_DM_PDF03.incumbent_solution`

source anchors: `DM_PDF03:p001:L001`, `DM_PDF03:p002:L001`, `DM_PDF03:p003:L001`, `DM_PDF03:p008:L001`, `DM_PDF04:p005:L002`, `DM_PDF04:p005:L006`


In [ ]:
# 힌트:
# - max 정수문제에서 LP relaxation 값 Z는 해당 노드의 upper bound다.
# - incumbent L은 현재까지 발견한 가장 좋은 정수가능해 값이다.
# - 소수값 v는 x <= floor(v), x >= ceil(v) 두 branch로 나눈다.
# - prune 사유는 infeasible, Z <= L, integer solution 발견으로 구분한다.
# - fractional solution은 incumbent로 갱신할 수 없다.


#### 문항별 시각화 학습자료 — 문제 1
- 목적: root LP relaxation, 첫 branch, 다음 branch 후보를 한 화면에서 확인한다.
- 체크포인트: `Z`는 upper bound이고, `L`은 정수가능해가 발견될 때만 갱신된다.
- 풀이용이므로 최종 정수해와 최종 목적값은 비워 둔다.


In [ ]:
# 시각화 업데이트: 문제 1 Branch-and-Bound 트리 뼈대
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["axes.unicode_minus"] = False

# 문제에서 이미 주어진 LP relaxation 정보만 표시한다. 최종 정수해는 표시하지 않는다.
nodes = {
    0: {"xy": (0.50, 0.82), "label": "Node 0\nLP x=(3.189, 4.324)\nZ=260.0\nbranch: x1"},
    1: {"xy": (0.25, 0.48), "label": "Node 1\nx1 <= 3\nLP x=(3.0, 4.4)\nZ=257.2\nnext: x2"},
    2: {"xy": (0.75, 0.48), "label": "Node 2\nx1 >= 4\nLP x=(4.0, 2.5)\nZ=215.0\nUpdate L? ____"},
    3: {"xy": (0.13, 0.16), "label": "Node 1-a\nx2 <= floor(4.4)\nstudent calc"},
    4: {"xy": (0.37, 0.16), "label": "Node 1-b\nx2 >= ceil(4.4)\nstudent calc"},
}
edges = [(0,1), (0,2), (1,3), (1,4)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
ax = axes[0]
for a, b in edges:
    xa, ya = nodes[a]["xy"]
    xb, yb = nodes[b]["xy"]
    ax.plot([xa, xb], [ya, yb], color="0.55", lw=1.6)
for nid, info in nodes.items():
    x, y = info["xy"]
    ax.scatter(x, y, s=980, color="#eef4ff", edgecolor="#2f5f9f", zorder=3)
    ax.text(x, y, info["label"], ha="center", va="center", fontsize=8)
ax.set_title("B&B tree scaffold: fill Z and L")
ax.axis("off")

# feasible region과 branch 기준선
ax = axes[1]
x1 = np.linspace(0, 5.5, 300)
y_a = (46 - 9*x1) / 4
y_b = (28 - 2*x1) / 5
upper = np.minimum(y_a, y_b)
upper = np.maximum(upper, 0)
ax.fill_between(x1, 0, upper, where=upper>=0, color="#d8efe4", alpha=0.8, label="LP feasible region")
ax.plot(x1, y_a, color="#26734d", label="9x1+4x2<=46")
ax.plot(x1, y_b, color="#5d6db3", label="2x1+5x2<=28")
ax.scatter([3.189, 3.0, 4.0], [4.324, 4.4, 2.5], color=["#d84a4a", "#f2a541", "#f2a541"], zorder=5)
ax.text(3.189, 4.324, " root Z", fontsize=9, va="bottom")
for v, lab, col in [(3, "x1<=3", "#d84a4a"), (4, "x1>=4", "#d84a4a"), (4, "x2<=4", "#555555"), (5, "x2>=5", "#555555")]:
    if lab.startswith("x1"):
        ax.axvline(v, color=col, ls="--", lw=1)
        ax.text(v+0.03, 0.2, lab, rotation=90, fontsize=8)
    else:
        ax.axhline(v, color=col, ls=":", lw=1)
        ax.text(0.1, v+0.05, lab, fontsize=8)
ax.set_xlim(0, 5.5)
ax.set_ylim(0, 6)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title("LP relaxation region and branch lines")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

print("관찰: 소수 LP 해는 정수해가 아니므로 branch 대상이다.")
print("체크: 각 노드에서 Z, L, prune 사유를 표에 직접 채워라.")


### 내 답안
- 변수 정의:
- 목적함수:
- 제약식:
- domain:
- Solver mapping:
- LP relaxation/논리 해석:
- 오답 진단:
- source anchor / node_id:


### 문제 2. 위성부품 0-1 배낭문제 — fractional bound와 include/exclude tree

한 항공우주 회사가 위성 탑재 부품 후보 6개 중 일부를 선택한다. 각 부품은 하나씩만 탑재할 수 있다. 총 중량은 34kg 이하여야 한다.

| 부품 j | 가치 | 무게 |
|---:|---:|---:|
| 1 | 42 | 12 |
| 2 | 30 | 8 |
| 3 | 25 | 7 |
| 4 | 24 | 9 |
| 5 | 18 | 6 |
| 6 | 15 | 5 |

Maximize

`z = 42x1 + 30x2 + 25x3 + 24x4 + 18x5 + 15x6`

subject to

`12x1 + 8x2 + 7x3 + 9x4 + 6x5 + 5x6 <= 34`

`x_j = 0 or 1`

요구:
1. `x_j`의 0-1 의미를 설명하라.
2. LP relaxation을 쓰라.
3. 가치/무게 비율을 계산하고 fractional knapsack bound를 구하는 순서를 설명하라.
4. fractional bound가 왜 0-1 정수문제의 upper bound가 되는지 설명하라.
5. 첫 branch를 `x1=1`과 `x1=0`으로 나누어라.
6. 각 branch에서 remaining capacity와 bound 계산 방식을 설명하라.
7. feasible integer solution을 발견했을 때 incumbent를 갱신하는 기준을 쓰라.
8. 어떤 branch를 bound로 prune할 수 있는지 설명하라.
9. LP relaxation의 fractional solution을 그대로 0-1 해로 쓰면 왜 틀리는지 설명하라.
10. Solver에서 bin option을 넣어야 하는 이유를 설명하라.
11. 오답 진단: “가치/무게 비율 순서대로 무조건 선택하면 0-1 knapsack 최적해다.” 이 답안의 오류를 설명하라.

node_id: `n_DM_PDF04.knapsack_problem`, `n_DM_PDF04.binary_variable`, `n_DM_PDF04.lp_relaxation`, `n_DM_PDF03.branch_and_bound`, `n_DM_PDF03.upper_lower_bound`

source anchors: `DM_PDF04:p008:L002`, `DM_PDF04:p009:L002`, `DM_PDF04:p009:L006`, `DM_PDF04:p010:L002`, `DM_PDF03:p001:L001`


In [ ]:
# 힌트:
# - max 정수문제에서 LP relaxation 값 Z는 해당 노드의 upper bound다.
# - incumbent L은 현재까지 발견한 가장 좋은 정수가능해 값이다.
# - 소수값 v는 x <= floor(v), x >= ceil(v) 두 branch로 나눈다.
# - prune 사유는 infeasible, Z <= L, integer solution 발견으로 구분한다.
# - fractional solution은 incumbent로 갱신할 수 없다.


#### 문항별 시각화 학습자료 — 문제 2
- 목적: 가치/무게 비율, 용량 34kg, fractional bound가 왜 상한인지 확인한다.
- 체크포인트: 비율 순서는 bound 계산용이지 0-1 최적해 증명이 아니다.
- 풀이용이므로 최종 선택 조합은 표시하지 않는다.


In [ ]:
# 시각화 업데이트: 문제 2 Knapsack ratio와 capacity scaffold
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

values = np.array([42, 30, 25, 24, 18, 15])
weights = np.array([12, 8, 7, 9, 6, 5])
items = np.arange(1, 7)
ratio = values / weights
order = np.argsort(-ratio)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
colors = ["#4c78a8" if i in order[:3] else "#bab0ac" for i in range(len(items))]
ax.bar(items, ratio, color=colors)
for i, r in zip(items, ratio):
    ax.text(i, r + 0.04, f"{r:.2f}", ha="center", fontsize=9)
ax.set_xticks(items)
ax.set_xlabel("item j")
ax.set_ylabel("value/weight")
ax.set_title("value/weight ratio for fractional bound")

ax = axes[1]
ax.barh([0], [34], color="#e6e6e6", edgecolor="0.4")
left = 0
for idx in order:
    take = min(weights[idx], max(0, 34-left))
    if take <= 0:
        break
    ax.barh([0], [take], left=[left], color="#86bc86", edgecolor="white")
    ax.text(left + take/2, 0, f"j{idx+1}\n{take:g}kg", ha="center", va="center", fontsize=8)
    left += take
ax.set_xlim(0, 36)
ax.set_yticks([])
ax.set_xlabel("capacity used (kg)")
ax.set_title("capacity 34kg fill scaffold by ratio")
plt.tight_layout()
plt.show()

ratio_table = pd.DataFrame({"item": items, "value": values, "weight": weights, "value_per_weight": ratio}).sort_values("value_per_weight", ascending=False)
display(ratio_table)
print("체크: fractional 조각이 나오면 upper bound이지 0-1 선택 조합이 아니다.")


### 내 답안
- 변수 정의:
- 목적함수:
- 제약식:
- domain:
- Solver mapping:
- LP relaxation/논리 해석:
- 오답 진단:
- source anchor / node_id:


### 문제 3. Branch-and-Bound 오답진단과 최적성 증명 — 주어진 tree 해석

다음은 어떤 max 정수계획문제의 branch-and-bound 탐색 결과 일부다. 모든 정수조건 변수는 `x1, x2`이다.

| 노드 | 추가 제약 | LP relaxation solution | Z | 상태 |
|---:|---|---|---:|---|
| 0 | 없음 | x1=3.4, x2=2.6 | 96.2 | fractional |
| 1 | x1 <= 3 | x1=3.0, x2=2.8 | 94.6 | fractional |
| 2 | x1 >= 4 | x1=4.0, x2=1.9 | 92.7 | fractional |
| 3 | x1 <= 3, x2 <= 2 | x1=3.0, x2=2.0 | 88.0 | integer |
| 4 | x1 <= 3, x2 >= 3 | infeasible | - | infeasible |
| 5 | x1 >= 4, x2 <= 1 | x1=4.0, x2=1.0 | 82.0 | integer |
| 6 | x1 >= 4, x2 >= 2 | x1=4.5, x2=2.0 | 91.5 | fractional |
| 7 | x1 >= 5, x2 >= 2 | infeasible | - | infeasible |
| 8 | x1 <= 4, x1 >= 4, x2 >= 2 | x1=4.0, x2=2.0 | 90.0 | integer |

요구:
1. root node의 LP relaxation value가 어떤 bound인지 설명하라.
2. node 0에서 `x1=3.4`를 기준으로 어떤 두 branch를 만들 수 있는지 쓰라.
3. node 3이 발견되었을 때 incumbent `L`을 어떻게 갱신하는지 설명하라.
4. node 4와 node 7의 pruning 사유를 쓰라.
5. node 5는 정수해이지만 최종해가 아닐 수 있는 이유를 설명하라.
6. node 6은 fractional인데 왜 계속 branch해야 하는지 설명하라.
7. node 8이 발견된 후 `L`을 갱신할지 판정하라.
8. 남은 open node가 없을 때 최적성을 어떻게 증명하는지 쓰라.
9. 다음 학생 답안을 진단하라: “LP relaxation 값이 가장 큰 node 0의 해 x1=3.4, x2=2.6을 반올림하면 최적 정수해다.”
10. 다음 학생 답안을 진단하라: “정수해가 하나 발견되면 바로 전체 최적해가 증명된다.”

node_id: `n_DM_PDF03.pruning_rules`, `n_DM_PDF03.incumbent_solution`, `n_DM_PDF03.node_selection`, `n_DM_PDF03.upper_lower_bound`

source anchors: `DM_PDF03:p001:L001`, `DM_PDF03:p002:L001`, `DM_PDF03:p008:L001`, `DM_PDF03:p009:L001`


In [ ]:
# 힌트:
# - max 정수문제에서 LP relaxation 값 Z는 해당 노드의 upper bound다.
# - incumbent L은 현재까지 발견한 가장 좋은 정수가능해 값이다.
# - 소수값 v는 x <= floor(v), x >= ceil(v) 두 branch로 나눈다.
# - prune 사유는 infeasible, Z <= L, integer solution 발견으로 구분한다.
# - fractional solution은 incumbent로 갱신할 수 없다.


#### 문항별 시각화 학습자료 — 문제 3
- 목적: 주어진 tree의 각 노드에서 `Z`, 상태, `L` 갱신 후보를 분리해 읽는다.
- 체크포인트: fractional node는 incumbent가 될 수 없고, infeasible node는 즉시 절단된다.
- 풀이용이므로 최종 판정 문장은 직접 작성한다.


In [ ]:
# 시각화 업데이트: 문제 3 주어진 B&B tree 읽기 scaffold
import matplotlib.pyplot as plt

nodes = {
    0: (0.50, 0.90, "0\nZ=96.2\nfractional"),
    1: (0.25, 0.68, "1\nZ=94.6\nfractional"),
    2: (0.75, 0.68, "2\nZ=92.7\nfractional"),
    3: (0.12, 0.46, "3\nZ=88\ninteger\nL?"),
    4: (0.38, 0.46, "4\ninfeasible"),
    5: (0.62, 0.46, "5\nZ=82\ninteger\nL?"),
    6: (0.88, 0.46, "6\nZ=91.5\nfractional"),
    7: (0.76, 0.24, "7\ninfeasible"),
    8: (0.98, 0.24, "8\nZ=90\ninteger\nL?"),
}
edges = [(0,1), (0,2), (1,3), (1,4), (2,5), (2,6), (6,7), (6,8)]
status_color = {"fractional":"#fff1cc", "integer":"#d8efe4", "infeasible":"#f4cccc"}

fig, ax = plt.subplots(figsize=(11, 5.2))
for a,b in edges:
    ax.plot([nodes[a][0], nodes[b][0]], [nodes[a][1], nodes[b][1]], color="0.5")
for nid, (x,y,label) in nodes.items():
    status = "infeasible" if "infeasible" in label else "integer" if "integer" in label else "fractional"
    ax.scatter(x, y, s=1300, color=status_color[status], edgecolor="0.25", zorder=3)
    ax.text(x, y, label, ha="center", va="center", fontsize=9)
ax.set_title("Given tree: read Z, status, and L update")
ax.axis("off")
plt.show()

print("체크: node 3, 5, 8 중 어느 노드가 L을 갱신하는지 순서대로 판단하라.")
print("체크: node 6은 fractional인데 왜 바로 버리면 안 되는지 Z와 L로 설명하라.")


### 내 답안
- 변수 정의:
- 목적함수:
- 제약식:
- domain:
- Solver mapping:
- LP relaxation/논리 해석:
- 오답 진단:
- source anchor / node_id:
